In [ ]:
# 03b_train_ensemble.ipynb

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize

from src.utils.config import load_config
from src.models.lightgbm_model import LightGBMModel
from src.models.randomforest_model import RandomForestMultiModel
from src.models.ensemble_model import EnsembleModel
from src.models.artifact import save_model_artifact

In [ ]:
# ==========================================
# 1. 환경 설정 및 OOF(Out-of-Fold) 데이터 로드
# ==========================================
cfg = load_config()
ref_date = cfg['project']['reference_date']
model_date = cfg['universe']['model_date']

# 경로 규칙에 따라 모델별 디렉토리 설정
training_base = Path(cfg['paths']['training_dir']) / model_date

print("📥 개별 모델의 OOF 예측 결과 로드 중...")
df_lgbm = pd.read_parquet(training_base / "lightgbm" / "predictions.parquet")
df_rf = pd.read_parquet(training_base / "randomforest" / "predictions.parquet")

# 타깃 및 정답 컬럼 추출
target_cols = [c for c in df_lgbm.columns if c.startswith('pred_')]
true_cols = [c.replace('pred_', 'true_') for c in target_cols]

# numpy 배열 변환 (학습 속도 극대화)
preds_lgbm = df_lgbm[target_cols].values
preds_rf = df_rf[target_cols].values
trues = df_lgbm[true_cols].values

# 유효한(NaN이 아닌) 데이터만 필터링
valid_mask = ~np.isnan(trues).any(axis=1)
preds_lgbm = preds_lgbm[valid_mask]
preds_rf = preds_rf[valid_mask]
trues = trues[valid_mask]

In [ ]:
# ==========================================
# 2. Scipy 기반 최적 가중치 탐색 (Optimized Blending)
# ==========================================
def objective(weights):
    """주어진 가중치에 대한 앙상블 RMSE 반환"""
    w1, w2 = weights
    blended = (w1 * preds_lgbm) + (w2 * preds_rf)
    mse = np.mean((blended - trues)**2)
    return np.sqrt(mse)

# 제약조건: 가중치의 합은 1.0 (w1 + w2 = 1)
cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
# 경계조건: 각 가중치는 0 이상 1 이하
bounds = [(0, 1), (0, 1)]
init_weights = [0.3, 0.7]

print("🔍 최적 가중치 탐색 시작 (SLSQP)...")
res = minimize(objective, init_weights, method='SLSQP', bounds=bounds, constraints=cons)

best_w1, best_w2 = res.x
print(f"✅ 최적 가중치: LightGBM = {best_w1:.4f} | RandomForest = {best_w2:.4f}")
print(f"   - 단일 LGBM RMSE: {objective([1, 0]):.6f}")
print(f"   - 단일 RF RMSE  : {objective([0, 1]):.6f}")
print(f"   - 🚀 앙상블 RMSE: {res.fun:.6f}")

In [ ]:
# ==========================================
# 3. EnsembleModel 인스턴스화 및 아티팩트 저장
# ==========================================
print("\n📦 학습된 개별 모델 로드 및 조립 중...")
lgbm_path = list((training_base / "lightgbm").glob("*.pkl"))[0]
rf_path = list((training_base / "randomforest").glob("*.pkl"))[0]

lgbm_model = LightGBMModel.load(str(lgbm_path))
rf_model = RandomForestMultiModel.load(str(rf_path))

# 래퍼 모델 조립
ensemble = EnsembleModel(
    model_version=f"v1_ens_{ref_date}",
    models=[lgbm_model, rf_model],
    weights=[best_w1, best_w2]
)

# 앙상블 모델 전용 디렉토리에 저장
ensemble_dir = training_base / "ensemble"
ensemble_dir.mkdir(parents=True, exist_ok=True)

save_model_artifact(
    model_name="ensemble",
    model_version=ensemble.model_version,
    model_object=ensemble,
    metadata={
        "weights": {"lightgbm": best_w1, "randomforest": best_w2},
        "ensemble_rmse": res.fun,
        "target_columns": ensemble.target_columns
    },
    model_dir=ensemble_dir
)
print(f"✅ 앙상블 모델 저장 완료: {ensemble_dir.name}/")

In [ ]:
print("\n📊 앙상블 검증 결과(OOF) 생성 및 저장 중...")

# 1. 가중치가 적용된 앙상블 예측값 계산 (OOF 데이터 활용)
# preds_lgbm, preds_rf는 앞선 단계에서 로드된 numpy 배열
blended_oof_preds = (best_w1 * preds_lgbm) + (best_w2 * preds_rf)

# 2. 저장용 DataFrame 구축 (df_lgbm의 구조를 복사하여 정답지와 메타데이터 유지)
# valid_mask는 앞서 NaN을 제거할 때 사용한 마스크
df_ensemble_oof = df_lgbm[valid_mask].copy()

# 3. 예측값 컬럼 업데이트 (pred_target_log_close_h1 등)
for i, col in enumerate(target_cols):
    df_ensemble_oof[col] = blended_oof_preds[:, i]

# 4. 앙상블 폴더 내에 predictions.parquet 저장
# 이 파일이 있어야 05_universe_selection.ipynb가 에러 없이 작동합니다.
ensemble_preds_path = ensemble_dir / "predictions.parquet"
df_ensemble_oof.to_parquet(ensemble_preds_path, index=False)

print(f"✅ 앙상블 검증 결과 저장 완료: {ensemble_preds_path}")